## Загрузка библиотек

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import pandas as pd
from tqdm import tqdm
import gc
import numpy as np

from batch_poisoning import StochasticDiscretePoisoner

## Загрузка и предобработка данных

In [6]:
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

batch_size = 256
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)

val_sample_size = 512
val_loader_fixed = torch.utils.data.DataLoader(valset, batch_size=val_sample_size, shuffle=True)
fixed_val_inputs, fixed_val_labels = next(iter(val_loader_fixed))
fixed_val_inputs = fixed_val_inputs.to(device)
fixed_val_labels = fixed_val_labels.to(device)

Using device: cuda


In [7]:
num_epochs = 40

poison_rates = np.geomspace(0.4, 0.001, num=5)

experiments = {}
for rate in poison_rates:
    exp_name = f"StochSquare_rate_{rate:.4f}"

    experiments[exp_name] = StochasticDiscretePoisoner(
        poison_fraction=rate,
        mu=8.0,
        sigma=1.0,
        apply_prob=1.0
    )


for exp_name, poisoner in experiments.items():
    print(f"\n{'='*50}")
    print(f"Starting Experiment: {exp_name}")
    print(f"{'='*50}")

    model = resnet18(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

    logs = {
        'step': [],
        'train_loss':[],
        'val_loss':[],
        'val_accuracy': [],
        'poison_fraction':[]
    }

    global_step = 0

    for epoch in range(num_epochs):
        model.train()
        loop = tqdm(trainloader, desc=f"[{exp_name}] Epoch {epoch+1}/{num_epochs}")

        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)

            inputs, labels, actual_poison_fraction = poisoner(inputs, labels, step=global_step)

            optimizer.zero_grad()
            train_outputs = model(inputs)
            train_loss = criterion(train_outputs, labels)
            train_loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                val_outputs = model(fixed_val_inputs)
                val_loss = criterion(val_outputs, fixed_val_labels)

                _, predicted = val_outputs.max(1)
                correct = predicted.eq(fixed_val_labels).sum().item()
                val_acc = correct / val_sample_size
            model.train()

            logs['step'].append(global_step)
            logs['train_loss'].append(train_loss.item())
            logs['val_loss'].append(val_loss.item())
            logs['val_accuracy'].append(val_acc)
            logs['poison_fraction'].append(actual_poison_fraction)

            global_step += 1

            loop.set_postfix(t_loss=train_loss.item(), v_loss=val_loss.item(), v_acc=val_acc, poison=actual_poison_fraction)

    df_logs = pd.DataFrame(logs)
    csv_name = f'resnet_cifar_{exp_name}_logs.csv'
    df_logs.to_csv(csv_name, index=False)
    print(f"Saved {exp_name} logs to {csv_name}")

    del model, criterion, optimizer, df_logs
    torch.cuda.empty_cache()
    gc.collect()

print("\nAll 5 experiments completed successfully!")


Starting Experiment: StochSquare_rate_0.4000


[StochSquare_rate_0.4000] Epoch 40/40: 100%|██████████| 196/196 [00:19<00:00, 10.10it/s, poison=0.4, t_loss=1.94, v_acc=0.734, v_loss=0.846]


Saved StochSquare_rate_0.4000 logs to resnet_cifar_StochSquare_rate_0.4000_logs.csv

Starting Experiment: StochSquare_rate_0.0894


[StochSquare_rate_0.0894] Epoch 40/40: 100%|██████████| 196/196 [00:18<00:00, 10.33it/s, poison=0.0875, t_loss=0.864, v_acc=0.76, v_loss=0.693]


Saved StochSquare_rate_0.0894 logs to resnet_cifar_StochSquare_rate_0.0894_logs.csv

Starting Experiment: StochSquare_rate_0.0200


...

Saved StochSquare_rate_0.0200 logs to resnet_cifar_StochSquare_rate_0.0200_logs.csv

Starting Experiment: StochSquare_rate_0.0045


[StochSquare_rate_0.0045] Epoch 40/40: 100%|██████████| 196/196 [00:19<00:00, 10.04it/s, poison=0, t_loss=0.469, v_acc=0.727, v_loss=0.777]


Saved StochSquare_rate_0.0045 logs to resnet_cifar_StochSquare_rate_0.0045_logs.csv

Starting Experiment: StochSquare_rate_0.0010


[StochSquare_rate_0.0010] Epoch 40/40: 100%|██████████| 196/196 [00:19<00:00, 10.25it/s, poison=0, t_loss=0.6, v_acc=0.771, v_loss=0.664]


Saved StochSquare_rate_0.0010 logs to resnet_cifar_StochSquare_rate_0.0010_logs.csv

All 5 experiments completed successfully!
